# GCN-MAPPO Adaptive Traffic Signal Control
## Manhattan 4×4 Grid — Kaggle Training Notebook

**Run cells in order. Do not skip any cell.**

In [ ]:
# ── CELL 1: Install SUMO ─────────────────────────────────────────────────────
# This takes 2-3 minutes on first run
import subprocess, sys

print('Installing SUMO...')
subprocess.run(['apt-get', 'install', '-y', 'sumo', 'sumo-tools', 'sumo-doc'],
               capture_output=True)

import os
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.append('/usr/share/sumo/tools')

# Verify
result = subprocess.run(['sumo', '--version'], capture_output=True, text=True)
if 'SUMO' in result.stdout:
    print('SUMO installed:', result.stdout.split('\n')[0])
else:
    print('ERROR: SUMO not found. Check output above.')
    print(result.stderr)

In [ ]:
# ── CELL 2: Set working directory ────────────────────────────────────────────
import os

# All your files should be uploaded as a Kaggle Dataset
# The dataset will be at /kaggle/input/YOUR-DATASET-NAME/
# We copy everything to /kaggle/working/ so we can write output files

DATASET_NAME = 'marl-traffic'  # ← change this to your Kaggle dataset name
INPUT_DIR    = f'/kaggle/input/{DATASET_NAME}'
WORK_DIR     = '/kaggle/working'

# Copy all files to working dir (input is read-only on Kaggle)
import shutil, glob
for f in glob.glob(f'{INPUT_DIR}/*'):
    dest = os.path.join(WORK_DIR, os.path.basename(f))
    if not os.path.exists(dest):
        shutil.copy2(f, dest)
        print(f'Copied: {os.path.basename(f)}')

os.chdir(WORK_DIR)
print(f'\nWorking directory: {os.getcwd()}')
print('Files present:')
for f in sorted(os.listdir('.')):
    print(f'  {f}')

In [ ]:
# ── CELL 3: Install Python dependencies ──────────────────────────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'numpy', '-q'])
print('Python packages ready')

In [ ]:
# ── CELL 4: Quick environment test ───────────────────────────────────────────
import os
os.environ['SUMO_HOME'] = '/usr/share/sumo'

import sys
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')

from gcn_encoder import REAL_JUNCTIONS, RAW_OBS_DIM, ACT_DIM
from traci_env import SUMOEnv, SUMO_CFG, _SCRIPTS_DIR
import numpy as np

print(f'SUMO files dir : {_SCRIPTS_DIR}')
print(f'run.sumocfg    : {SUMO_CFG}')
print(f'Exists         : {os.path.exists(SUMO_CFG)}')
print()

# Quick 50-step test
print('Running 50-step environment test...')
env = SUMOEnv(warmup=50, ep_duration=100)
obs = env.reset()
rewards = []
for _ in range(50):
    acts = {jid: np.random.randint(0, ACT_DIM[jid]) for jid in REAL_JUNCTIONS}
    obs, rews, done, info = env.step(acts)
    rewards.extend(rews.values())
    if done: break
env.close()
print(f'Reward range: min={min(rewards):.3f}  max={max(rewards):.3f}  mean={np.mean(rewards):.3f}')
print('Environment test PASSED' if max(rewards) <= 0 else 'WARNING: check reward range')

In [ ]:
# ── CELL 5: TRAINING ─────────────────────────────────────────────────────────
import os, sys
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')

from traci_env import train

# ── Training config ───────────────────────────────────────────────────────────
# Kaggle GPU session = 12 hours, CPU = 9 hours
# Each episode = 3600 simulation steps ≈ 3-5 min on Kaggle CPU
# Recommended: 200 episodes ≈ 10-15 hours on CPU
#              500 episodes ≈ faster on GPU (T4)

import torch
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 200    # adjust based on Kaggle session time
print(f'Device: {DEVICE}')
print(f'Episodes: {EPISODES}')
print()

# Stage 1: Train on high density (peak hour) — hardest first
ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/kaggle/working/ckpt_high',
    use_gui     = False,
    curriculum  = False,
)

In [ ]:
# ── CELL 6 (optional): Curriculum training ───────────────────────────────────
# Run this only if Cell 5 completed and you have time remaining
import os, sys
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')

from traci_env import train
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

ctrl = train(
    n_episodes   = 300,
    device       = DEVICE,
    save_dir     = '/kaggle/working/ckpt_curriculum',
    resume_from  = '/kaggle/working/ckpt_high',   # resume from Stage 1
    use_gui      = False,
    curriculum   = True,
)

In [ ]:
# ── CELL 7: Evaluate and plot results ────────────────────────────────────────
import os, sys, pandas as pd, matplotlib.pyplot as plt
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')

from traci_env import _evaluate
from mappo_atsc import MAPPOController
import torch

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_DIR = '/kaggle/working/ckpt_high_best'  # or ckpt_curriculum_best

if os.path.exists(CKPT_DIR):
    ctrl = MAPPOController(device=DEVICE)
    ctrl.load(CKPT_DIR)

    print('Evaluating...')
    for density in ['low', 'high']:
        print(f'\n  density={density}:')
        _evaluate(ctrl, n_reps=3, use_gui=False, density=density)
else:
    print(f'Checkpoint not found at {CKPT_DIR} — run Cell 5 first')

# Plot training curve
log_path = '/kaggle/working/ckpt_high/training_log.csv'
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.plot(df['episode'], df['avg_reward'])
    ax1.set_title('Average Reward per Episode')
    ax1.set_xlabel('Episode'); ax1.set_ylabel('Reward')
    ax2.plot(df['episode'], df['avg_travel_time'], color='orange')
    ax2.set_title('Average Travel Time per Episode')
    ax2.set_xlabel('Episode'); ax2.set_ylabel('Travel Time (s)')
    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curve.png', dpi=150)
    plt.show()
    print('Plot saved: training_curve.png')